# ДЗ1, шаг 2. Охота на баги

**Ваш вариант: seed 42.** В код ниже подсажено **3** семантических ошибки.
У других студентов свой набор, поэтому чужой ответ вам не поможет.

**Что значит «семантическая».** Код запускается, обучается и не падает. Ошибки меняют не
работоспособность, а то, *что* вычисляется. Такие баги в реальной работе самые дорогие:
модель учится, метрики выглядят правдоподобно, а результат неверный.

**Что нужно сделать.**

1. Прогоните проверки и посмотрите, какие упали.
2. Найдите каждую ошибку в коде.
3. По каждой напишите три строки: **где** сидит (функция и строка), **какой механизм**
   ломает и **почему** это меняет результат, **как** вы её исправили.
4. Почините и добейтесь зелёного прогона.
5. Обучите исправленную модель ячейкой в конце и приложите кривую потерь и пример
   сгенерированного текста.

**Что не считается ответом.** «Здесь опечатка», «неправильная ось», «поменял на другое
значение». Нужен механизм. Так выглядит принимаемый ответ:

> Делят на `d_k` вместо корня из `d_k`. Дисперсия логитов растёт вместе с размерностью,
> softmax насыщается: распределение вырождается в почти one-hot, производная softmax
> становится близкой к нулю, и градиенты через внимание почти не проходят.



In [ ]:
# В Colab torch уже стоит. Ячейка на случай локального запуска.
import sys
try:
    import torch
    print('torch', torch.__version__)
except ImportError:
    !{sys.executable} -m pip install -q torch
    import torch
    print('torch', torch.__version__)

## Код с подсаженными ошибками

Читайте внимательно: docstring-и описывают, как должно быть. Расхождение между
описанием и кодом — и есть подсказка.

Правьте прямо в этой ячейке и выполняйте её заново.

In [ ]:
from __future__ import annotations

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class GPTConfig:
    vocab_size: int = 256
    context_size: int = 64
    d_model: int = 64
    n_heads: int = 4
    n_layers: int = 2
    d_ff: int | None = None      # по умолчанию 4 * d_model
    dropout: float = 0.0

    def __post_init__(self):
        if self.d_ff is None:
            self.d_ff = 4 * self.d_model
        if self.d_model % self.n_heads:
            raise ValueError('d_model должно делиться на n_heads без остатка')

    @property
    def d_head(self) -> int:
        return self.d_model // self.n_heads


# --- позиционное кодирование --------------------------------------------------

def sinusoidal_positional_encoding(length: int, d_model: int) -> torch.Tensor:
    """Синусоидальное кодирование из «Attention is All You Need», форма (length, d_model).

    Чётные каналы — синус, нечётные — косинус; частота падает с номером канала.
    """
    position = torch.arange(1, length + 1, dtype=torch.float32).unsqueeze(1)
    channel = torch.arange(0, d_model, 2, dtype=torch.float32)
    div_term = torch.exp(-math.log(10000.0) * channel / d_model)
    pe = torch.zeros(length, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


# --- нормализация -------------------------------------------------------------

class LayerNorm(nn.Module):
    """Нормализация по признакам внутри токена (последняя ось), а не по батчу."""

    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))
        self.bias = nn.Parameter(torch.zeros(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=0, keepdim=True)
        var = x.var(dim=0, keepdim=True, unbiased=False)
        normalized = (x - mean) / torch.sqrt(var + self.eps)
        return self.weight * normalized + self.bias


# --- внимание -----------------------------------------------------------------

def split_heads(x: torch.Tensor, n_heads: int) -> torch.Tensor:
    """(B, L, d_model) -> (B, n_heads, L, d_head).

    Сначала разворачиваем последнюю ось в (n_heads, d_head), только потом переставляем
    оси. Обратный порядок перемешает каналы между головами.
    """
    batch, length, d_model = x.shape
    d_head = d_model // n_heads
    x = x.view(batch, length, n_heads, d_head)
    return x.transpose(1, 2)


def merge_heads(x: torch.Tensor) -> torch.Tensor:
    """(B, n_heads, L, d_head) -> (B, L, d_model). Обратная операция к split_heads."""
    batch, n_heads, length, d_head = x.shape
    x = x.transpose(1, 2).contiguous()
    return x.view(batch, length, n_heads * d_head)


def scaled_dot_product_attention(query: torch.Tensor, key: torch.Tensor,
                                 value: torch.Tensor,
                                 mask: torch.Tensor | None = None,
                                 dropout: nn.Module | None = None):
    """Внимание: softmax(QKᵀ / sqrt(d_k)) V. Возвращает (выход, веса внимания).

    Деление на корень из d_k держит дисперсию логитов независимой от размерности:
    без него softmax насыщается и градиенты гаснут. Маску применяем ДО softmax,
    подставляя -inf, чтобы запрещённые позиции получили нулевой вес.
    """
    d_k = query.size(-1)
    scores = query @ key.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = torch.softmax(scores, dim=-2)
    if dropout is not None:
        weights = dropout(weights)
    return weights @ value, weights


class MultiHeadAttention(nn.Module):
    """Многоголовое причинное внимание."""

    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.n_heads = cfg.n_heads
        self.w_q = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.w_k = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.w_v = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.w_o = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.dropout = nn.Dropout(cfg.dropout)
        causal = torch.tril(torch.ones(cfg.context_size, cfg.context_size))
        self.register_buffer('causal_mask', causal.view(1, 1, cfg.context_size, cfg.context_size))

    def forward(self, x: torch.Tensor):
        length = x.size(1)
        q = split_heads(self.w_q(x), self.n_heads)
        k = split_heads(self.w_k(x), self.n_heads)
        v = split_heads(self.w_v(x), self.n_heads)
        mask = self.causal_mask[:, :, :length, :length]
        out, weights = scaled_dot_product_attention(q, k, v, mask, self.dropout)
        return self.w_o(merge_heads(out)), weights


# --- полносвязный подслой -----------------------------------------------------

class FeedForward(nn.Module):
    """Две проекции с нелинейностью между ними.

    Без нелинейности композиция двух Linear схлопывается в один линейный слой,
    и подслой перестаёт что-либо добавлять.
    """

    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.up = nn.Linear(cfg.d_model, cfg.d_ff)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        hidden = F.gelu(self.up(x))
        return self.dropout(self.down(hidden))


# --- блок и модель ------------------------------------------------------------

class TransformerBlock(nn.Module):
    """Pre-norm блок: x + attn(norm(x)), затем x + ffn(norm(x))."""

    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.norm1 = LayerNorm(cfg.d_model)
        self.attn = MultiHeadAttention(cfg)
        self.norm2 = LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg)

    def forward(self, x: torch.Tensor):
        attn_out, weights = self.attn(self.norm1(x))
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x, weights


class MiniGPT(nn.Module):
    """Декодер-трансформер: эмбеддинги, позиционное кодирование, блоки, голова."""

    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg
        self.token_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.register_buffer('pos_emb',
                             sinusoidal_positional_encoding(cfg.context_size, cfg.d_model))
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList(TransformerBlock(cfg) for _ in range(cfg.n_layers))
        self.norm_out = LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        length = idx.size(1)
        if length > self.cfg.context_size:
            raise ValueError(f'последовательность {length} длиннее контекста {self.cfg.context_size}')
        x = self.token_emb(idx) + self.pos_emb[:length]
        x = self.drop(x)
        attentions = []
        for block in self.blocks:
            x, weights = block(x)
            attentions.append(weights)
        logits = self.lm_head(self.norm_out(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, attentions


@torch.no_grad()
def generate(model: MiniGPT, idx: torch.Tensor, max_new_tokens: int,
             temperature: float = 1.0, top_k: int | None = None) -> torch.Tensor:
    """Авторегрессионная генерация. model.eval() обязателен: иначе работает dropout."""
    was_training = model.training
    model.eval()
    for _ in range(max_new_tokens):
        window = idx[:, -model.cfg.context_size:]
        logits, _, _ = model(window)
        logits = logits[:, -1, :] / max(temperature, 1e-8)
        if top_k is not None:
            kth = torch.topk(logits, top_k, dim=-1).values[:, -1:]
            logits = logits.masked_fill(logits < kth, float('-inf'))
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_token], dim=1)
    if was_training:
        model.train()
    return idx

## Проверки

Те же, что в шаге 1. Выполните ячейку — она ничего не печатает.

In [ ]:
import math

import torch
import torch.nn.functional as F

CHECKS = []


def check(title, hint):
    def wrap(fn):
        CHECKS.append(dict(name=fn.__name__, title=title, hint=hint, fn=fn))
        return fn
    return wrap


def close(a, b, tol=1e-5):
    return abs(float(a) - float(b)) <= tol


def cfg(m, **kw):
    base = dict(vocab_size=32, context_size=16, d_model=32, n_heads=4, n_layers=2, dropout=0.0)
    base.update(kw)
    return m.GPTConfig(**base)


# --- позиционное кодирование --------------------------------------------------

@check('позиционное кодирование совпадает с формулой',
       'чётные каналы должны быть синусом, нечётные — косинусом одной и той же частоты')
def positional_encoding_matches_formula(m):
    length, d_model = 8, 16
    pe = m.sinusoidal_positional_encoding(length, d_model)
    assert pe.shape == (length, d_model), f'форма {tuple(pe.shape)}, ожидалась {(length, d_model)}'
    for pos in range(length):
        for i in range(0, d_model, 2):
            w = math.exp(-math.log(10000.0) * i / d_model)
            assert close(pe[pos, i], math.sin(pos * w)), f'канал {i} на позиции {pos}'
            assert close(pe[pos, i + 1], math.cos(pos * w)), f'канал {i + 1} на позиции {pos}'


@check('нулевая позиция закодирована нулевой позицией',
       'проверьте, с какого числа начинается отсчёт позиций')
def positional_encoding_starts_at_zero_position(m):
    pe = m.sinusoidal_positional_encoding(4, 8)
    assert torch.allclose(pe[0, 0::2], torch.zeros(4), atol=1e-6), 'синусы на позиции 0 не нулевые'
    assert torch.allclose(pe[0, 1::2], torch.ones(4), atol=1e-6), 'косинусы на позиции 0 не единичные'


# --- нормализация -------------------------------------------------------------

@check('LayerNorm нормирует каждый токен отдельно',
       'посмотрите, по какой оси считаются среднее и дисперсия')
def layernorm_normalizes_each_token_separately(m):
    norm = m.LayerNorm(16)
    x = torch.randn(4, 6, 16) * 5 + 3
    x[0] *= 100
    y = norm(x)
    assert torch.allclose(y.mean(dim=-1), torch.zeros(4, 6), atol=1e-4), 'среднее по признакам не ноль'
    assert torch.allclose(y.var(dim=-1, unbiased=False), torch.ones(4, 6), atol=1e-3), \
        'дисперсия по признакам не единица'


@check('нормировка одного примера не зависит от остальных в батче',
       'если статистики считаются по батчу, получается batch norm, а не layer norm')
def layernorm_is_independent_of_other_batch_elements(m):
    norm = m.LayerNorm(16)
    a = torch.randn(1, 3, 16)
    b = torch.randn(1, 3, 16) * 50
    alone = norm(a)
    together = norm(torch.cat([a, b], dim=0))[:1]
    assert torch.allclose(alone, together, atol=1e-5), 'соседний пример батча изменил результат'


# --- головы -------------------------------------------------------------------

@check('merge_heads в точности обращает split_heads',
       'одна из двух функций переставляет оси не так, как другая')
def split_merge_roundtrip(m):
    x = torch.randn(2, 5, 32)
    back = m.merge_heads(m.split_heads(x, 4))
    assert back.shape == x.shape, f'форма после сборки {tuple(back.shape)}, ожидалась {tuple(x.shape)}'
    assert torch.allclose(back, x, atol=1e-6), 'содержимое после сборки не совпало с исходным'


@check('голова получает свой кусок каналов',
       'нужную форму дают два способа; только один кладёт в голову h её собственные каналы')
def split_heads_shape_and_layout(m):
    x = torch.randn(2, 5, 32)
    heads = m.split_heads(x, 4)
    assert heads.shape == (2, 4, 5, 8), f'форма {tuple(heads.shape)}, ожидалась (2, 4, 5, 8)'
    for h in range(4):
        assert torch.allclose(heads[:, h], x[..., h * 8:(h + 1) * 8], atol=1e-6), \
            f'в голове {h} лежат не её каналы'


# --- внимание -----------------------------------------------------------------

@check('внимание совпадает с эталонной реализацией PyTorch',
       'расходятся масштаб логитов, порядок Q и K либо ось softmax')
def attention_matches_reference_implementation(m):
    torch.manual_seed(0)
    q, k, v = (torch.randn(2, 4, 6, 8) for _ in range(3))
    out, _ = m.scaled_dot_product_attention(q, k, v)
    assert torch.allclose(out, F.scaled_dot_product_attention(q, k, v), atol=1e-5)


@check('веса внимания суммируются к единице по ключам',
       'каждая строка матрицы внимания обязана быть распределением')
def attention_weights_sum_to_one_over_keys(m):
    torch.manual_seed(0)
    q, k, v = (torch.randn(2, 4, 6, 8) for _ in range(3))
    _, w = m.scaled_dot_product_attention(q, k, v)
    assert w.shape == (2, 4, 6, 6), f'форма весов {tuple(w.shape)}, ожидалась (2, 4, 6, 6)'
    assert torch.allclose(w.sum(dim=-1), torch.ones(2, 4, 6), atol=1e-5)


@check('масштаб логитов зависит от корня из d_k',
       'при d_k = 64 деление на d_k вместо корня даёт заметно другие веса')
def attention_scale_depends_on_sqrt_d_k(m):
    torch.manual_seed(1)
    q, k, v = (torch.randn(1, 1, 12, 64) for _ in range(3))
    _, w = m.scaled_dot_product_attention(q, k, v)
    expected = torch.softmax(q @ k.transpose(-2, -1) / math.sqrt(64), dim=-1)
    assert torch.allclose(w, expected, atol=1e-5)


@check('причинная маска обнуляет запрещённые позиции',
       'важно, на каком шаге применяется маска и какое значение подставляется')
def attention_respects_causal_mask(m):
    torch.manual_seed(2)
    q, k, v = (torch.randn(1, 1, 5, 8) for _ in range(3))
    mask = torch.tril(torch.ones(5, 5)).view(1, 1, 5, 5)
    _, w = m.scaled_dot_product_attention(q, k, v, mask)
    upper = w[0, 0].triu(diagonal=1)
    assert torch.allclose(upper, torch.zeros_like(upper), atol=1e-7), \
        'будущие позиции получили ненулевой вес'
    assert torch.allclose(w.sum(dim=-1), torch.ones(1, 1, 5), atol=1e-5), \
        'после маскирования веса больше не суммируются к единице'


@check('модель не подглядывает в будущее',
       'меняем будущие токены — предсказания на прошлых позициях обязаны не измениться')
def no_future_leak_in_model(m):
    torch.manual_seed(2)
    model = m.MiniGPT(cfg(m)).eval()
    a = torch.randint(0, 32, (1, 12))
    b = a.clone()
    b[0, 7:] = torch.randint(0, 32, (5,))
    with torch.no_grad():
        la, _, _ = model(a)
        lb, _, _ = model(b)
    assert torch.allclose(la[:, :7], lb[:, :7], atol=1e-5)


# --- полносвязный подслой и блок ----------------------------------------------

@check('FFN нелинеен',
       'две проекции подряд без нелинейности схлопываются в одно линейное преобразование')
def ffn_is_nonlinear(m):
    torch.manual_seed(3)
    ffn = m.FeedForward(cfg(m))
    a, b = torch.randn(1, 1, 32), torch.randn(1, 1, 32)
    gap = (ffn(a + b) - ffn(a) - ffn(b) + ffn(torch.zeros_like(a))).abs().max()
    assert gap > 1e-3, 'подслой ведёт себя как линейный'


@check('блок сохраняет остаточный путь',
       'обнуляем выходы подслоёв — блок обязан вернуть вход без изменений')
def block_keeps_residual_path(m):
    torch.manual_seed(4)
    block = m.TransformerBlock(cfg(m))
    with torch.no_grad():
        block.attn.w_o.weight.zero_()
        block.ffn.down.weight.zero_()
        block.ffn.down.bias.zero_()
    x = torch.randn(2, 6, 32)
    out, _ = block(x)
    assert torch.allclose(out, x, atol=1e-5)


# --- модель и генерация -------------------------------------------------------

@check('модель отдаёт логиты, потери и карты внимания нужных форм',
       'проверьте, что forward возвращает кортеж (логиты, потери, список карт внимания)')
def model_shapes_and_loss(m):
    torch.manual_seed(5)
    model = m.MiniGPT(cfg(m))
    idx = torch.randint(0, 32, (3, 10))
    logits, loss, attentions = model(idx, targets=idx)
    assert logits.shape == (3, 10, 32), f'логиты {tuple(logits.shape)}, ожидалось (3, 10, 32)'
    assert loss.item() > 0, 'потери не положительны'
    assert len(attentions) == 2, f'карт внимания {len(attentions)}, ожидалось 2 (по слою)'
    assert attentions[0].shape == (3, 4, 10, 10), f'карта {tuple(attentions[0].shape)}'


@check('при top_k=1 генерация совпадает с argmax',
       'посмотрите, от какого конца отсортированных логитов берётся порог')
def generate_respects_top_k_one(m):
    torch.manual_seed(5)
    model = m.MiniGPT(cfg(m)).eval()
    idx = torch.randint(0, 32, (1, 4))
    out1 = m.generate(model, idx, max_new_tokens=3, top_k=1)
    out2 = m.generate(model, idx, max_new_tokens=3, top_k=1)
    assert torch.equal(out1, out2), 'два одинаковых вызова дали разное'
    with torch.no_grad():
        logits, _, _ = model(idx)
    assert out1[0, 4].item() == logits[0, -1].argmax().item(), 'выбран не самый вероятный токен'


@check('генерация переводит модель в режим инференса',
       'что делает dropout, если модель осталась в режиме обучения')
def generate_switches_model_to_eval(m):
    torch.manual_seed(6)
    model = m.MiniGPT(cfg(m, dropout=0.5)).train()
    idx = torch.randint(0, 32, (1, 4))
    a = m.generate(model, idx, max_new_tokens=4, top_k=1)
    b = m.generate(model, idx, max_new_tokens=4, top_k=1)
    assert torch.equal(a, b), 'генерация нестабильна от вызова к вызову'


@check('модель обучается: потери падают на одной последовательности',
       'интеграционная проверка; если падает только она, ищите ошибку в градиентном пути')
def model_can_overfit_tiny_sequence(m):
    torch.manual_seed(7)
    model = m.MiniGPT(cfg(m, n_layers=1))
    idx = torch.randint(0, 32, (2, 12))
    x, y = idx[:, :-1], idx[:, 1:]
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    _, first, _ = model(x, targets=y)
    loss = first
    for _ in range(120):
        opt.zero_grad()
        _, loss, _ = model(x, targets=y)
        loss.backward()
        opt.step()
    assert loss.item() < first.item() * 0.5, \
        f'потери почти не изменились: {first.item():.3f} -> {loss.item():.3f}'


# --- запуск -------------------------------------------------------------------

def run(target, verbose=True):
    """Прогоняет все проверки по объекту с реализацией. Возвращает список падений."""
    failures = []
    for c in CHECKS:
        try:
            c['fn'](target)
        except NotImplementedError as exc:
            failures.append((c, f'не реализовано: {exc}'))
            mark, note = 'ПУСТО ', 'функция ещё не дописана'
        except AssertionError as exc:
            failures.append((c, str(exc) or 'проверка не прошла'))
            mark, note = 'ОШИБКА', (str(exc) or '') + f'\n         подсказка: {c["hint"]}'
        except Exception as exc:                       # noqa: BLE001
            failures.append((c, f'{type(exc).__name__}: {exc}'))
            mark, note = 'СБОЙ  ', f'{type(exc).__name__}: {exc}'
        else:
            mark, note = 'ok    ', ''
        if verbose:
            print(f'{mark} {c["title"]}')
            if note:
                print(f'         {note}')
    if verbose:
        total, bad = len(CHECKS), len(failures)
        print(f'\nпройдено {total - bad} из {total}')
        if not bad:
            print('Все проверки зелёные.')
    return failures

## Прогон

Упавшие проверки указывают, *что* нарушено. Где именно — ищете вы.

In [ ]:
# Собираем объект с вашей реализацией и прогоняем проверки.
# Проверки открыты: их код лежит в ячейке выше, и при приёме прогоняются ровно они.
import types

implementation = types.SimpleNamespace(
    GPTConfig=GPTConfig,
    sinusoidal_positional_encoding=sinusoidal_positional_encoding,
    LayerNorm=LayerNorm,
    split_heads=split_heads,
    merge_heads=merge_heads,
    scaled_dot_product_attention=scaled_dot_product_attention,
    MultiHeadAttention=MultiHeadAttention,
    FeedForward=FeedForward,
    TransformerBlock=TransformerBlock,
    MiniGPT=MiniGPT,
    generate=generate,
)

failures = run(implementation)

## Ваши ответы

Заполните таблицу — по строке на каждую найденную ошибку. Ошибок в вашем варианте 3.

| # | Функция и строка | Механизм поломки | Как исправили |
|---|---|---|---|
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |

## Обучение исправленной модели

Короткий цикл на игрушечных данных: он показывает, что после исправлений модель
действительно учится. Приложите к отчёту график потерь и пример генерации.

In [ ]:
# Игрушечный корпус: учим модель на повторяющемся шаблоне.
# Данные нарочно простые — задача ячейки показать, что потери падают, а не обучить LLM.
import torch

torch.manual_seed(0)
text = 'алгоритм внимания смотрит на все позиции сразу. ' * 200
vocab = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(vocab)}
itos = {i: ch for ch, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text])

cfg = GPTConfig(vocab_size=len(vocab), context_size=64, d_model=128,
                n_heads=4, n_layers=2, dropout=0.1)
model = MiniGPT(cfg)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)

def batch(size=32):
    pos = torch.randint(0, len(data) - cfg.context_size - 1, (size,))
    x = torch.stack([data[p:p + cfg.context_size] for p in pos])
    y = torch.stack([data[p + 1:p + cfg.context_size + 1] for p in pos])
    return x, y

history = []
for step in range(600):
    x, y = batch()
    _, loss, _ = model(x, targets=y)
    opt.zero_grad()
    loss.backward()
    opt.step()
    history.append(loss.item())
    if step % 100 == 0:
        print(f'шаг {step:4d}  потери {loss.item():.4f}')

print(f'итог: {history[0]:.3f} -> {history[-1]:.3f}')

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3))
plt.plot(history)
plt.xlabel('шаг обучения')
plt.ylabel('потери')
plt.title('Кривая потерь после исправления багов')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Пример генерации. Если баг в top-k или в режиме модели остался,
# это видно здесь: текст либо вырождается в один символ, либо не воспроизводится.
start = torch.tensor([[stoi['а']]])
out = generate(model, start, max_new_tokens=200, temperature=0.8, top_k=10)
print(''.join(itos[int(i)] for i in out[0]))

## Что приложить к отчёту

1. Этот ноутбук с зелёным прогоном проверок.
2. Заполненную таблицу ответов с механизмом каждой поломки.
3. Кривую потерь и пример генерации.